# Zero-Shot Classification for T-Mobile Review Analysis
## Advanced ML Analysis Without Training

This notebook uses Facebook's BART model fine-tuned on MNLI (Multi-Genre Natural Language Inference) to perform zero-shot classification on customer reviews. This approach provides more accurate categorization than keyword matching while requiring no training data.

**Model**: `facebook/bart-large-mnli`  
**Dataset**: 6,170 T-Mobile Trustpilot reviews  
**Time**: ~10-15 min on GPU, ~1-2 hours on CPU


## 1. Setup and Imports


In [11]:
# Install required packages (uncomment and run once)
!pip install transformers torch accelerate scipy tqdm



    pytz (>dev)
         ~^

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [12]:
import json
import os
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime
import numpy as np
from tqdm.auto import tqdm

# ML imports
from transformers import pipeline
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")


ImportError: cannot import name 'is_mistral_common_available' from 'transformers.utils' (/opt/anaconda3/lib/python3.11/site-packages/transformers/utils/__init__.py)

## 2. Load All Reviews (6k+ reviews from API)


In [ ]:
def load_all_reviews(reviews_dir='tmobile_reviews'):
    """Load all reviews from JSON files"""
    all_reviews = []
    review_ids = set()  # Track unique reviews
    
    for json_file in sorted(Path(reviews_dir).glob('*.json')):
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                reviews = json.load(f)
                
                # Remove duplicates
                for review in reviews:
                    if review['id'] not in review_ids:
                        review_ids.add(review['id'])
                        all_reviews.append(review)
        except Exception as e:
            print(f"Error loading {json_file}: {e}")
    
    print(f"✅ Loaded {len(all_reviews)} unique reviews")
    return all_reviews

reviews = load_all_reviews()

# Show sample
print(f"\nSample review:")
print(f"Rating: {reviews[0]['rating']}")
print(f"Text: {reviews[0]['text'][:200]}...")


## 3. Initialize Zero-Shot Classifier

Using `facebook/bart-large-mnli` - a BART model fine-tuned on Multi-Genre Natural Language Inference. This model can classify text into **any categories we define** without additional training.


In [ ]:
# Initialize zero-shot classifier
device = 0 if torch.cuda.is_available() else -1
print(f"Loading model on device: {'GPU' if device == 0 else 'CPU'}")

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device
)

print("✅ Model loaded successfully!")


## 4. Define Classification Categories

We'll classify reviews into multiple categories relevant to telecom customer experience.


In [ ]:
# Issue categories
ISSUE_CATEGORIES = [
    "billing and pricing problems",
    "poor network coverage and connectivity",
    "bad customer service experience",
    "device and technical issues",
    "contract and cancellation problems",
    "misleading promotions and false advertising",
    "account and security issues"
]

# Churn indicators
CHURN_LABELS = [
    "customer wants to leave or cancel service",
    "customer is satisfied and staying"
]

# Sentiment categories (more nuanced than just rating)
SENTIMENT_LABELS = [
    "very frustrated and angry",
    "disappointed but calm",
    "neutral or mixed feelings",
    "satisfied and happy",
    "extremely positive and grateful"
]

print(f"Issue categories: {len(ISSUE_CATEGORIES)}")
print(f"Churn labels: {len(CHURN_LABELS)}")
print(f"Sentiment labels: {len(SENTIMENT_LABELS)}")


## 5. Test on Sample Review


In [ ]:
def classify_review(review_text, categories, multi_label=True, threshold=0.5):
    """Classify a single review"""
    try:
        # Truncate very long reviews to avoid memory issues
        text = review_text[:1024] if len(review_text) > 1024 else review_text
        
        result = classifier(
            text,
            candidate_labels=categories,
            multi_label=multi_label
        )
        
        if multi_label:
            # Return all labels above threshold
            return [
                {"label": label, "score": score}
                for label, score in zip(result['labels'], result['scores'])
                if score >= threshold
            ]
        else:
            # Return top label
            return {"label": result['labels'][0], "score": result['scores'][0]}
    
    except Exception as e:
        print(f"Error classifying review: {e}")
        return None

# Test on a sample review
sample_review = reviews[0]['text']
print("Testing classifier on sample review...\n")
print(f"Review: {sample_review[:200]}...\n")

issues = classify_review(sample_review, ISSUE_CATEGORIES, multi_label=True, threshold=0.3)
print(f"Issues detected: {issues}\n")

churn = classify_review(sample_review, CHURN_LABELS, multi_label=False)
print(f"Churn prediction: {churn}\n")

sentiment = classify_review(sample_review, SENTIMENT_LABELS, multi_label=False)
print(f"Sentiment: {sentiment}")


In [ ]:
# Classify all reviews (this will take time!)
# Start with a subset for testing, then run on full dataset
USE_SUBSET = True  # Set to False to process all reviews
SUBSET_SIZE = 500  # Test with 500 reviews first

reviews_to_process = reviews[:SUBSET_SIZE] if USE_SUBSET else reviews
print(f"Processing {len(reviews_to_process)} reviews...\n")

analyzed_reviews = []

for review in tqdm(reviews_to_process, desc="Analyzing reviews"):
    text = review['text']
    
    # Skip very short reviews
    if len(text) < 50:
        continue
    
    analysis = {
        'id': review['id'],
        'text': text,
        'rating': review['rating'],
        'timestamp': review['timestamp'],
        'issues': classify_review(text, ISSUE_CATEGORIES, multi_label=True, threshold=0.3),
        'churn_prediction': classify_review(text, CHURN_LABELS, multi_label=False),
        'sentiment': classify_review(text, SENTIMENT_LABELS, multi_label=False)
    }
    
    analyzed_reviews.append(analysis)

print(f"\n✅ Analyzed {len(analyzed_reviews)} reviews")


## 7. Generate Actionable Insights


In [ ]:
def generate_insights(analyzed_reviews):
    """Generate actionable insights from classified reviews"""
    
    # Count issues
    issue_counts = Counter()
    for review in analyzed_reviews:
        if review['issues']:
            for issue in review['issues']:
                issue_counts[issue['label']] += 1
    
    # Count churn predictions
    churn_count = sum(
        1 for r in analyzed_reviews 
        if r['churn_prediction'] and 
        'leave' in r['churn_prediction']['label'].lower() and
        r['churn_prediction']['score'] > 0.6
    )
    
    # Sentiment distribution
    sentiment_counts = Counter()
    for review in analyzed_reviews:
        if review['sentiment']:
            sentiment_counts[review['sentiment']['label']] += 1
    
    # Calculate metrics
    total_reviews = len(analyzed_reviews)
    avg_rating = np.mean([r['rating'] for r in analyzed_reviews])
    negative_reviews = sum(1 for r in analyzed_reviews if r['rating'] <= 2)
    
    # Churn risk score (0-100)
    churn_risk_score = min(100, int((churn_count / total_reviews) * 100 * 1.5))
    
    # Top issues
    top_issues = [
        {
            'category': issue,
            'count': count,
            'percentage': f"{(count/total_reviews)*100:.1f}%"
        }
        for issue, count in issue_counts.most_common(7)
    ]
    
    # Generate actionable insights
    insights = []
    
    # Top issue insights
    if top_issues:
        top_issue = top_issues[0]
        insights.append({
            'title': f"Critical: {top_issue['category'].title()}",
            'description': f"Mentioned in {top_issue['percentage']} of reviews - the most common complaint",
            'action': f"Prioritize immediate investigation and improvement of {top_issue['category']}",
            'severity': 'critical'
        })
    
    # Churn insight
    if churn_risk_score > 50:
        insights.append({
            'title': f"High Churn Risk: {churn_count} customers",
            'description': f"{(churn_count/total_reviews)*100:.1f}% of reviews indicate intent to cancel service",
            'action': "Implement proactive retention campaigns and address root causes immediately",
            'severity': 'critical'
        })
    
    # Sentiment insight
    frustrated_count = sum(
        count for label, count in sentiment_counts.items()
        if 'frustrated' in label.lower() or 'angry' in label.lower()
    )
    if frustrated_count > total_reviews * 0.3:
        insights.append({
            'title': "Customer Frustration at Critical Levels",
            'description': f"{(frustrated_count/total_reviews)*100:.1f}% of reviews show extreme frustration",
            'action': "Review customer service protocols and empower agents to resolve issues faster",
            'severity': 'high'
        })
    
    # Package results
    results = {
        'summary': {
            'total_reviews_analyzed': total_reviews,
            'analysis_method': 'zero-shot-classification',
            'model': 'facebook/bart-large-mnli',
            'timestamp': datetime.now().isoformat()
        },
        'metrics': {
            'churn_risk_score': churn_risk_score,
            'at_risk_customers': churn_count,
            'average_rating': round(avg_rating, 2),
            'negative_reviews_count': negative_reviews,
            'negative_reviews_percentage': round((negative_reviews/total_reviews)*100, 1)
        },
        'top_issues': top_issues,
        'sentiment_distribution': [
            {'sentiment': k, 'count': v, 'percentage': f"{(v/total_reviews)*100:.1f}%"}
            for k, v in sentiment_counts.most_common()
        ],
        'actionable_insights': insights,
        'detailed_reviews': analyzed_reviews[:100]  # Include top 100 for reference
    }
    
    return results

insights_data = generate_insights(analyzed_reviews)

# Display summary
print("\n" + "="*60)
print("ZERO-SHOT CLASSIFICATION INSIGHTS")
print("="*60)
print(f"\n📊 Churn Risk Score: {insights_data['metrics']['churn_risk_score']}/100")
print(f"⚠️  At-Risk Customers: {insights_data['metrics']['at_risk_customers']}")
print(f"⭐ Average Rating: {insights_data['metrics']['average_rating']}/5")
print(f"👎 Negative Reviews: {insights_data['metrics']['negative_reviews_percentage']}%")

print("\n🔥 Top Issues:")
for i, issue in enumerate(insights_data['top_issues'][:5], 1):
    print(f"   {i}. {issue['category']}: {issue['count']} reviews ({issue['percentage']})")

print("\n💡 Key Insights:")
for i, insight in enumerate(insights_data['actionable_insights'], 1):
    print(f"   {i}. {insight['title']}")
    print(f"      → {insight['action']}")


In [ ]:
# Save insights for the API
output_file = 'trustpilot_insights_zeroshot.json'

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(insights_data, f, indent=2, ensure_ascii=False)

print(f"✅ Insights saved to {output_file}")
print(f"\nFile size: {os.path.getsize(output_file) / 1024:.1f} KB")

# Also save all analyzed reviews for further analysis
detailed_output = 'analyzed_reviews_zeroshot.json'
with open(detailed_output, 'w', encoding='utf-8') as f:
    json.dump(analyzed_reviews, f, indent=2, ensure_ascii=False)

print(f"✅ Detailed analysis saved to {detailed_output}")


## 9. Compare with Keyword Baseline


In [ ]:
# Load keyword-based results if available
try:
    with open('trustpilot_insights.json', 'r') as f:
        keyword_insights = json.load(f)
    
    print("📊 COMPARISON: Zero-Shot vs Keyword-Based")
    print("=" * 60)
    
    print("\nChurn Risk Score:")
    print(f"  Keyword-based: {keyword_insights.get('churn_risk_score', 'N/A')}")
    print(f"  Zero-shot:     {insights_data['metrics']['churn_risk_score']}")
    
    print("\nAt-Risk Customers:")
    print(f"  Keyword-based: {keyword_insights.get('at_risk_customers', 'N/A')}")
    print(f"  Zero-shot:     {insights_data['metrics']['at_risk_customers']}")
    
    print("\n✨ Advantages of Zero-Shot:")
    print("  ✓ Understands context and nuance")
    print("  ✓ No manual keyword curation needed")
    print("  ✓ Captures implicit sentiment")
    print("  ✓ Multi-label classification (one review → multiple issues)")
    print("  ✓ More accurate confidence scores")
    
except FileNotFoundError:
    print("⚠️  Keyword-based results not found. Run analyze_trustpilot_insights.py first.")


## 🎯 Next Steps

### For Hackathon:
1. **Run on full dataset**: Set `USE_SUBSET = False` and process all 6k reviews
2. **Update API**: Modify `insights_routes.py` to serve zero-shot results
3. **Update dashboard**: Display improved insights in React frontend

### Hackathon Talking Points:
- ✅ **No training required** - leverages transfer learning from MNLI
- ✅ **Flexible** - classify into any categories without retraining
- ✅ **Context-aware** - understands nuance better than keywords
- ✅ **Multi-label** - one review tagged with multiple issues
- ✅ **Confidence scores** - quantifies prediction certainty
- ✅ **Production-ready** - deploy as-is for real-time classification

### Optional Improvements:
- Try different models (`facebook/bart-large`, `valhalla/distilbart-mnli-12-3`)
- Experiment with label descriptions for better accuracy
- Add batch processing for faster inference
- Cache results to avoid reprocessing
